Jakob should pip install "scikit-learn"

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from pathlib import Path
from datetime import datetime
#import file with the cleaned data

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
file_path = DATA_DIR / "DMI_API_3stations.csv"

generation_path = DATA_DIR / 'combined_data_positive_gen.csv'

In [ ]:
#Create df
df_combined = pd.read_csv(generation_path, 
                     delimiter=',', 
                     decimal='.', 
                     parse_dates=['ts'],  # Your timestamp column name
                     index_col='ts')

# Merge on timestamp
df_train = pd.merge(df_combined[['ts', 'radiation']],
                    df_combined[['ts', 'pv_power']],
                    on='ts',
                    how='inner')

# Remove missing values
df_train = df_train.dropna(subset=['radiation', 'pv_power'])


In [ ]:
# Fit model
X = df_train[['radiation']]
y = df_train['pv_power']

pv_model = LinearRegression()
pv_model.fit(X, y)

# Predict for all weather timestamps
weather_df['pv_pred'] = pv_model.predict(weather_df[['radiation']])

In [ ]:
wind_weather_df['ts'] = pd.to_datetime(wind_weather_df['ts'])
wind_power_df['ts'] = pd.to_datetime(wind_power_df['ts'])

df_train_wind = pd.merge(wind_weather_df[['ts', 'wind_speed']],
                         wind_power_df[['ts', 'wind_power']],
                         on='ts',
                         how='inner')

df_train_wind = df_train_wind.dropna(subset=['wind_speed', 'wind_power'])

df_train_wind['wind_speed_cubed'] = df_train_wind['wind_speed']**3
wind_weather_df['wind_speed_cubed'] = wind_weather_df['wind_speed']**3

Xw = df_train_wind[['wind_speed_cubed']]
yw = df_train_wind['wind_power']

wind_model = LinearRegression()
wind_model.fit(Xw, yw)

wind_weather_df['wind_pred'] = wind_model.predict(
    wind_weather_df[['wind_speed_cubed']]
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Convert timestamps
weather_df['ts'] = pd.to_datetime(weather_df['ts'])
pv_df['ts'] = pd.to_datetime(pv_df['ts'])

# Merge on timestamp
df_train = pd.merge(
    weather_df[['ts', 'radiation']],
    pv_df[['ts', 'pv_power']],
    on='ts',
    how='inner'
)

# Remove missing values
df_train = df_train.dropna(subset=['radiation', 'pv_power'])

# Fit model
X = df_train[['radiation']]
y = df_train['pv_power']

pv_model = LinearRegression()
pv_model.fit(X, y)

# Predict on training data for evaluation
df_train['pv_pred'] = pv_model.predict(X)

# Evaluate model
pv_r2 = r2_score(df_train['pv_power'], df_train['pv_pred'])
pv_mae = mean_absolute_error(df_train['pv_power'], df_train['pv_pred'])
pv_rmse = mean_squared_error(df_train['pv_power'], df_train['pv_pred']) ** 0.5

print("PV model")
print("Slope:", pv_model.coef_[0])
print("Intercept:", pv_model.intercept_)
print("R²:", pv_r2)
print("MAE:", pv_mae)
print("RMSE:", pv_rmse)

# Predict for all weather timestamps
weather_df['pv_pred'] = pv_model.predict(weather_df[['radiation']])

# Optional: no negative PV output
weather_df['pv_pred'] = weather_df['pv_pred'].clip(lower=0)

# Visual check 1: measured vs predicted scatter
plt.figure(figsize=(6, 6))
plt.scatter(df_train['pv_power'], df_train['pv_pred'], alpha=0.6)
plt.xlabel("Measured PV power")
plt.ylabel("Predicted PV power")
plt.title("PV model: measured vs predicted")

# 1:1 reference line
min_val = min(df_train['pv_power'].min(), df_train['pv_pred'].min())
max_val = max(df_train['pv_power'].max(), df_train['pv_pred'].max())
plt.plot([min_val, max_val], [min_val, max_val])
plt.show()

# Visual check 2: time series comparison on training timestamps
plt.figure(figsize=(12, 4))
plt.plot(df_train['ts'], df_train['pv_power'], label='Measured PV power')
plt.plot(df_train['ts'], df_train['pv_pred'], label='Predicted PV power')
plt.xlabel("Time")
plt.ylabel("PV power")
plt.title("PV model: measured and predicted over time")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Convert timestamps
wind_weather_df['ts'] = pd.to_datetime(wind_weather_df['ts'])
wind_power_df['ts'] = pd.to_datetime(wind_power_df['ts'])

# Merge on timestamp
df_train_wind = pd.merge(
    wind_weather_df[['ts', 'wind_speed']],
    wind_power_df[['ts', 'wind_power']],
    on='ts',
    how='inner'
)

# Remove missing values
df_train_wind = df_train_wind.dropna(subset=['wind_speed', 'wind_power'])

# Create transformed variable
df_train_wind['wind_speed_cubed'] = df_train_wind['wind_speed']**3
wind_weather_df['wind_speed_cubed'] = wind_weather_df['wind_speed']**3

# Fit model
Xw = df_train_wind[['wind_speed_cubed']]
yw = df_train_wind['wind_power']

wind_model = LinearRegression()
wind_model.fit(Xw, yw)

# Predict on training data for evaluation
df_train_wind['wind_pred'] = wind_model.predict(Xw)

# Evaluate model
wind_r2 = r2_score(df_train_wind['wind_power'], df_train_wind['wind_pred'])
wind_mae = mean_absolute_error(df_train_wind['wind_power'], df_train_wind['wind_pred'])
wind_rmse = mean_squared_error(df_train_wind['wind_power'], df_train_wind['wind_pred']) ** 0.5

print("Wind model")
print("Slope:", wind_model.coef_[0])
print("Intercept:", wind_model.intercept_)
print("R²:", wind_r2)
print("MAE:", wind_mae)
print("RMSE:", wind_rmse)

# Predict for all weather timestamps
wind_weather_df['wind_pred'] = wind_model.predict(
    wind_weather_df[['wind_speed_cubed']]
)

# Optional: clip to physical range
wind_weather_df['wind_pred'] = wind_weather_df['wind_pred'].clip(lower=0, upper=rated_power)

# Visual check 1: measured vs predicted scatter
plt.figure(figsize=(6, 6))
plt.scatter(df_train_wind['wind_power'], df_train_wind['wind_pred'], alpha=0.6)
plt.xlabel("Measured wind power")
plt.ylabel("Predicted wind power")
plt.title("Wind model: measured vs predicted")

# 1:1 reference line
min_val = min(df_train_wind['wind_power'].min(), df_train_wind['wind_pred'].min())
max_val = max(df_train_wind['wind_power'].max(), df_train_wind['wind_pred'].max())
plt.plot([min_val, max_val], [min_val, max_val])
plt.show()

# Visual check 2: time series comparison on training timestamps
plt.figure(figsize=(12, 4))
plt.plot(df_train_wind['ts'], df_train_wind['wind_power'], label='Measured wind power')
plt.plot(df_train_wind['ts'], df_train_wind['wind_pred'], label='Predicted wind power')
plt.xlabel("Time")
plt.ylabel("Wind power")
plt.title("Wind model: measured and predicted over time")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()